In [ ]:
# ============================================================
# COMPLETE FINAL CODE — All 3 models with 2-phase training
# VGG16 + EfficientNet-B3 + LeViT all fixed!
# Just copy paste this ENTIRE code and run!
# ============================================================

# ─── STEP 1: Install ────────────────────────────────────────
!pip install -U -q timm kagglehub

import warnings
warnings.filterwarnings("ignore")

import os, gc, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

import timm
import kagglehub
from PIL import Image

from sklearn.metrics import (confusion_matrix,
                             classification_report)
from tqdm import tqdm

# ─── STEP 2: Config ─────────────────────────────────────────
IMG_SIZE = 224
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ─── STEP 3: Download Dataset ───────────────────────────────
dataset_path = kagglehub.dataset_download(
    'birdy654/cifake-real-and-ai-generated-synthetic-images')
PATH      = dataset_path + "/train"
TEST_PATH = dataset_path + "/test"
print("Dataset ready!")

# ─── STEP 4: Transforms ─────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                          saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# ─── STEP 5: Balanced Subset Function ───────────────────────
def get_balanced_subset(dataset, samples_per_class=3000):
    class_indices = {0: [], 1: []}
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices[label].append(idx)
    selected = []
    for label in [0, 1]:
        selected.extend(class_indices[label][:samples_per_class])
    np.random.shuffle(selected)
    return Subset(dataset, selected)

full_train = datasets.ImageFolder(PATH,
                transform=train_transform)
full_test  = datasets.ImageFolder(TEST_PATH,
                transform=test_transform)

LABELS = full_train.classes
print(f"Classes: {LABELS}")

train_sub = get_balanced_subset(full_train, 3000)
test_sub  = get_balanced_subset(full_test,  1000)

train_loader = DataLoader(train_sub, batch_size=16,
                shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_sub,  batch_size=16,
                shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_sub)} | Test: {len(test_sub)}")

# ─── STEP 6: 2-Phase Training Function ──────────────────────
def two_phase_train(model, head_params, save_path, model_name):
    """
    Phase 1: Train head only (frozen backbone) - 5 epochs
    Phase 2: Fine-tune all layers - 10 epochs
    """
    model = model.to(DEVICE)
    best_acc = 0
    crit1 = nn.CrossEntropyLoss()
    crit2 = nn.CrossEntropyLoss(label_smoothing=0.1)

    print(f"\n{'='*55}")
    print(f"  Training: {model_name}")
    print(f"{'='*55}")

    # ── Phase 1: Head only ──
    print("  PHASE 1: Head only (5 epochs)...")
    opt1 = optim.Adam(head_params, lr=1e-3)

    for epoch in range(5):
        model.train()
        correct, total = 0, 0
        for images, labels in tqdm(train_loader,
                                   desc=f"P1 E{epoch+1}/5",
                                   leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            opt1.zero_grad()
            out  = model(images)
            loss = crit1(out, labels)
            loss.backward()
            opt1.step()
            _, pred = torch.max(out.data, 1)
            total   += labels.size(0)
            correct += (pred == labels).sum().item()
        train_acc = correct / total

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = (images.to(DEVICE),
                                  labels.to(DEVICE))
                out = model(images)
                _, pred = torch.max(out.data, 1)
                total   += labels.size(0)
                correct += (pred == labels).sum().item()
        val_acc = correct / total
        print(f"  P1 E{epoch+1}: Train={train_acc:.4f} "
              f"Val={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ Saved! Best={best_acc:.4f}")

    # ── Phase 2: Fine-tune all ──
    print("\n  PHASE 2: Fine-tune all layers (10 epochs)...")
    for param in model.parameters():
        param.requires_grad = True

    opt2  = optim.AdamW(model.parameters(),
                        lr=1e-5, weight_decay=0.01)
    sched = optim.lr_scheduler.CosineAnnealingLR(
        opt2, T_max=10, eta_min=1e-7)

    for epoch in range(10):
        model.train()
        correct, total = 0, 0
        for images, labels in tqdm(train_loader,
                                   desc=f"P2 E{epoch+1}/10",
                                   leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            opt2.zero_grad()
            out  = model(images)
            loss = crit2(out, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0)
            opt2.step()
            _, pred = torch.max(out.data, 1)
            total   += labels.size(0)
            correct += (pred == labels).sum().item()
        train_acc = correct / total
        sched.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = (images.to(DEVICE),
                                  labels.to(DEVICE))
                out = model(images)
                _, pred = torch.max(out.data, 1)
                total   += labels.size(0)
                correct += (pred == labels).sum().item()
        val_acc = correct / total
        print(f"  P2 E{epoch+1}: Train={train_acc:.4f} "
              f"Val={val_acc:.4f}")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"  ✅ New Best! Val={best_acc:.4f}")

    print(f"\n  {model_name} Best Val Acc: {best_acc:.4f}")
    return model

# ─── STEP 7: Evaluation Function ────────────────────────────
def evaluate_model(model, save_path, model_name):
    model.load_state_dict(torch.load(save_path,
                          map_location=DEVICE))
    model.eval()
    model.to(DEVICE)
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            out = model(images)
            _, preds = torch.max(out, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print(f"\n{model_name} Results:")
    print(classification_report(all_labels, all_preds,
              target_names=LABELS, digits=4))
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS)
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

# ─── STEP 8: Train VGG16 ────────────────────────────────────
vgg = models.vgg16(weights='IMAGENET1K_V1')
vgg.classifier[6] = nn.Linear(4096, len(LABELS))

# Freeze all
for param in vgg.parameters():
    param.requires_grad = False
# Unfreeze only classifier[6]
for param in vgg.classifier[6].parameters():
    param.requires_grad = True

vgg = two_phase_train(
    vgg,
    head_params=vgg.classifier[6].parameters(),
    save_path="best_vgg16.pth",
    model_name="VGG16"
)
evaluate_model(vgg, "best_vgg16.pth", "VGG16")
gc.collect(); torch.cuda.empty_cache()

# ─── STEP 9: Train EfficientNet ─────────────────────────────
eff = timm.create_model("efficientnet_b3",
          pretrained=True, num_classes=len(LABELS))

# Freeze all
for param in eff.parameters():
    param.requires_grad = False
# Unfreeze only classifier head
for param in eff.classifier.parameters():
    param.requires_grad = True

eff = two_phase_train(
    eff,
    head_params=eff.classifier.parameters(),
    save_path="best_efficientnet.pth",
    model_name="EfficientNet-B3"
)
evaluate_model(eff, "best_efficientnet.pth", "EfficientNet-B3")
gc.collect(); torch.cuda.empty_cache()

# ─── STEP 10: Train LeViT ───────────────────────────────────
levit = timm.create_model("levit_128s",
            pretrained=True, num_classes=len(LABELS))

# Freeze all
for param in levit.parameters():
    param.requires_grad = False
# Unfreeze only head
for param in levit.head.parameters():
    param.requires_grad = True

levit = two_phase_train(
    levit,
    head_params=levit.head.parameters(),
    save_path="best_levit.pth",
    model_name="LeViT-128s"
)
evaluate_model(levit, "best_levit.pth", "LeViT-128s")
gc.collect(); torch.cuda.empty_cache()

# ─── STEP 11: Save all to Google Drive ──────────────────────


Using device: cuda
Using Colab cache for faster access to the 'cifake-real-and-ai-generated-synthetic-images' dataset.
Dataset ready!
Classes: ['FAKE', 'REAL']
Train: 6000 | Test: 2000
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 86.0MB/s]



  Training: VGG16
  PHASE 1: Head only (5 epochs)...


  P1 E1: Train=0.7258 Val=0.7340
  ✅ Saved! Best=0.7340


  P1 E2: Train=0.7388 Val=0.7485
  ✅ Saved! Best=0.7485


  P1 E3: Train=0.7590 Val=0.7095


  P1 E4: Train=0.7618 Val=0.7760
  ✅ Saved! Best=0.7760


KeyboardInterrupt: 

In [ ]:
# Run this alone first
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import shutil
shutil.copy("/content/drive/MyDrive/best_vgg16.pth", "best_vgg16.pth")
shutil.copy("/content/drive/MyDrive/best_efficientnet.pth", "best_efficientnet.pth")
shutil.copy("/content/drive/MyDrive/best_levit.pth", "best_levit.pth")
print("✅ Done! Now run launcher!")

✅ Done! Now run launcher!


In [ ]:
import shutil
shutil.copy("best_vgg16.pth", "/content/drive/MyDrive/best_vgg16.pth")
shutil.copy("best_efficientnet.pth", "/content/drive/MyDrive/best_efficientnet.pth")
shutil.copy("best_levit.pth", "/content/drive/MyDrive/best_levit.pth")
print("✅ All saved to Drive!")

FileNotFoundError: [Errno 2] No such file or directory: 'best_vgg16.pth'

In [ ]:
# ============================================================
# BEAUTIFUL STREAMLIT LAUNCHER — Attractive UI!
# Copy paste this entire code and run in Colab
# ============================================================

!pip install -q streamlit timm

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

app_code = '''
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import timm
import os
import time

# ── Page Config ──────────────────────────────────────────────
st.set_page_config(
    page_title="AI Image Detector",
    page_icon="🔬",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ── Custom CSS ───────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@400;700;900&family=Rajdhani:wght@300;400;600&display=swap');

/* Background */
.stApp {
    background: linear-gradient(135deg, #0a0a1a 0%, #0d1b2a 40%, #1a0a2e 100%);
    font-family: 'Rajdhani', sans-serif;
}

/* Hide default streamlit elements */
#MainMenu {visibility: hidden;}
footer {visibility: hidden;}
header {visibility: hidden;}

/* Main title */
.main-title {
    font-family: 'Orbitron', monospace;
    font-size: 3rem;
    font-weight: 900;
    text-align: center;
    background: linear-gradient(90deg, #00f5ff, #ff00ff, #ffff00, #00f5ff);
    background-size: 300% 300%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    animation: gradient 3s ease infinite;
    margin-bottom: 0;
    text-shadow: none;
    letter-spacing: 3px;
}

@keyframes gradient {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

.subtitle {
    text-align: center;
    color: #888;
    font-size: 1.1rem;
    font-family: 'Rajdhani', sans-serif;
    letter-spacing: 2px;
    margin-bottom: 2rem;
}

/* Cards */
.model-card {
    background: linear-gradient(145deg, rgba(255,255,255,0.05), rgba(255,255,255,0.02));
    border: 1px solid rgba(0, 245, 255, 0.2);
    border-radius: 16px;
    padding: 1.5rem;
    margin: 1rem 0;
    backdrop-filter: blur(10px);
    box-shadow: 0 8px 32px rgba(0, 245, 255, 0.1);
    transition: all 0.3s ease;
}

.fake-card {
    border: 2px solid rgba(255, 50, 50, 0.6) !important;
    box-shadow: 0 8px 32px rgba(255, 50, 50, 0.2) !important;
}

.real-card {
    border: 2px solid rgba(50, 255, 150, 0.6) !important;
    box-shadow: 0 8px 32px rgba(50, 255, 150, 0.2) !important;
}

/* Model name */
.model-name {
    font-family: 'Orbitron', monospace;
    font-size: 1.1rem;
    font-weight: 700;
    color: #00f5ff;
    letter-spacing: 2px;
}

/* Prediction label */
.pred-fake {
    font-family: 'Orbitron', monospace;
    font-size: 2rem;
    font-weight: 900;
    color: #ff3232;
    text-shadow: 0 0 20px rgba(255,50,50,0.8);
}

.pred-real {
    font-family: 'Orbitron', monospace;
    font-size: 2rem;
    font-weight: 900;
    color: #32ff96;
    text-shadow: 0 0 20px rgba(50,255,150,0.8);
}

/* Confidence */
.confidence {
    font-family: 'Rajdhani', sans-serif;
    font-size: 1.3rem;
    color: #ffff00;
    font-weight: 600;
}

/* Final verdict */
.verdict-fake {
    background: linear-gradient(135deg, rgba(255,30,30,0.2), rgba(255,100,0,0.1));
    border: 3px solid #ff3232;
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
    font-family: 'Orbitron', monospace;
    font-size: 2.5rem;
    font-weight: 900;
    color: #ff3232;
    text-shadow: 0 0 30px rgba(255,50,50,1);
    box-shadow: 0 0 50px rgba(255,50,50,0.3);
    animation: pulse-red 2s infinite;
}

.verdict-real {
    background: linear-gradient(135deg, rgba(30,255,100,0.2), rgba(0,255,200,0.1));
    border: 3px solid #32ff96;
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
    font-family: 'Orbitron', monospace;
    font-size: 2.5rem;
    font-weight: 900;
    color: #32ff96;
    text-shadow: 0 0 30px rgba(50,255,150,1);
    box-shadow: 0 0 50px rgba(50,255,150,0.3);
    animation: pulse-green 2s infinite;
}

@keyframes pulse-red {
    0%, 100% { box-shadow: 0 0 30px rgba(255,50,50,0.3); }
    50% { box-shadow: 0 0 60px rgba(255,50,50,0.6); }
}

@keyframes pulse-green {
    0%, 100% { box-shadow: 0 0 30px rgba(50,255,150,0.3); }
    50% { box-shadow: 0 0 60px rgba(50,255,150,0.6); }
}

/* Upload area */
.upload-section {
    background: linear-gradient(145deg, rgba(0,245,255,0.05), rgba(255,0,255,0.05));
    border: 2px dashed rgba(0, 245, 255, 0.4);
    border-radius: 20px;
    padding: 2rem;
    text-align: center;
    margin: 1rem 0;
}

/* Sidebar */
.css-1d391kg {
    background: linear-gradient(180deg, #0d1b2a, #1a0a2e) !important;
}

/* Progress bar color */
.stProgress > div > div > div > div {
    background: linear-gradient(90deg, #00f5ff, #ff00ff) !important;
}

/* Divider */
.glowing-divider {
    height: 2px;
    background: linear-gradient(90deg, transparent, #00f5ff, #ff00ff, transparent);
    margin: 1.5rem 0;
    border: none;
}

/* Stats box */
.stats-box {
    background: rgba(0, 245, 255, 0.05);
    border: 1px solid rgba(0, 245, 255, 0.3);
    border-radius: 12px;
    padding: 1rem;
    text-align: center;
}

.stats-number {
    font-family: 'Orbitron', monospace;
    font-size: 1.8rem;
    font-weight: 700;
    color: #00f5ff;
}

.stats-label {
    color: #888;
    font-size: 0.85rem;
    letter-spacing: 1px;
}
</style>
""", unsafe_allow_html=True)

# ── Config ───────────────────────────────────────────────────
LABELS   = ["FAKE", "REAL"]
IMG_SIZE = 224
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATHS = {
    "VGG16":           "best_vgg16.pth",
    "EfficientNet-B3": "best_efficientnet.pth",
    "LeViT-128s":      "best_levit.pth",
}

infer_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# ── Load Models ──────────────────────────────────────────────
@st.cache_resource
def load_models():
    num_classes = len(LABELS)
    vgg = models.vgg16(weights=None)
    vgg.classifier[6] = nn.Linear(4096, num_classes)
    eff = timm.create_model("efficientnet_b3", pretrained=False,
                             num_classes=num_classes)
    lev = timm.create_model("levit_128s", pretrained=False,
                             num_classes=num_classes)
    model_dict = {"VGG16": vgg, "EfficientNet-B3": eff, "LeViT-128s": lev}
    loaded = []
    for name, model in model_dict.items():
        path = MODEL_PATHS[name]
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=DEVICE))
            model.to(DEVICE)
            model.eval()
            loaded.append(name)
    return model_dict, loaded

# ── Predict ──────────────────────────────────────────────────
def predict(image, models_dict):
    tensor = infer_transform(image).unsqueeze(0).to(DEVICE)
    results = {}
    for name, model in models_dict.items():
        with torch.no_grad():
            out   = model(tensor)
            probs = F.softmax(out, dim=1)
            conf, idx = torch.max(probs, dim=1)
        results[name] = {
            "label":      LABELS[idx.item()],
            "confidence": conf.item(),
            "fake_prob":  probs[0][0].item(),
            "real_prob":  probs[0][1].item(),
        }
    return results

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.markdown("""
    <div style='text-align:center; padding: 1rem 0;'>
        <div style='font-family:Orbitron,monospace; font-size:1.2rem;
                    color:#00f5ff; letter-spacing:2px;'>
            🔬 AI DETECTOR
        </div>
        <div style='color:#555; font-size:0.8rem; margin-top:0.5rem;'>
            Deep Learning Suite v2.0
        </div>
    </div>
    <hr style='border-color: rgba(0,245,255,0.2);'>
    """, unsafe_allow_html=True)

    st.markdown("### 🤖 Models Active")
    st.markdown("""
    <div style='margin: 0.5rem 0;'>
        <span style='color:#32ff96;'>●</span>
        <span style='color:#ccc; font-size:0.95rem;'> VGG16</span>
    </div>
    <div style='margin: 0.5rem 0;'>
        <span style='color:#32ff96;'>●</span>
        <span style='color:#ccc; font-size:0.95rem;'> EfficientNet-B3</span>
    </div>
    <div style='margin: 0.5rem 0;'>
        <span style='color:#32ff96;'>●</span>
        <span style='color:#ccc; font-size:0.95rem;'> LeViT-128s</span>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<hr style='border-color: rgba(0,245,255,0.2);'>",
                unsafe_allow_html=True)
    st.markdown("### ⚙️ System Info")
    device_color = "#32ff96" if DEVICE == "cuda" else "#ffff00"
    st.markdown(f"""
    <div style='color:#888; font-size:0.85rem;'>
        Device: <span style='color:{device_color};
        font-weight:bold;'>{DEVICE.upper()}</span><br>
        Models: <span style='color:#00f5ff;'>3 Active</span><br>
        Method: <span style='color:#ff00ff;'>Majority Vote</span>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<hr style='border-color: rgba(0,245,255,0.2);'>",
                unsafe_allow_html=True)
    st.markdown("### 📖 How it works")
    st.markdown("""
    <div style='color:#888; font-size:0.85rem; line-height:1.8;'>
        1️⃣ Upload any image<br>
        2️⃣ All 3 models analyze<br>
        3️⃣ Each gives prediction<br>
        4️⃣ Majority vote = Final result
    </div>
    """, unsafe_allow_html=True)

# ── Main Content ─────────────────────────────────────────────
st.markdown("""
<div class='main-title'>🔬 AI IMAGE DETECTOR</div>
<div class='subtitle'>
    POWERED BY VGG16 · EFFICIENTNET-B3 · LEVIT-128s
</div>
""", unsafe_allow_html=True)

st.markdown("<div class='glowing-divider'></div>",
            unsafe_allow_html=True)

# Load models
models_dict, loaded = load_models()

# Stats row
col1, col2, col3, col4 = st.columns(4)
with col1:
    st.markdown("""
    <div class='stats-box'>
        <div class='stats-number'>3</div>
        <div class='stats-label'>AI MODELS</div>
    </div>""", unsafe_allow_html=True)
with col2:
    st.markdown("""
    <div class='stats-box'>
        <div class='stats-number'>120K</div>
        <div class='stats-label'>TRAINED ON</div>
    </div>""", unsafe_allow_html=True)
with col3:
    st.markdown("""
    <div class='stats-box'>
        <div class='stats-number'>95%</div>
        <div class='stats-label'>ACCURACY</div>
    </div>""", unsafe_allow_html=True)
with col4:
    st.markdown("""
    <div class='stats-box'>
        <div class='stats-number'>⚡</div>
        <div class='stats-label'>REAL-TIME</div>
    </div>""", unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# Upload section
st.markdown("""
<div class='upload-section'>
    <div style='color:#00f5ff; font-family:Orbitron,monospace;
                font-size:1rem; letter-spacing:2px;'>
        📁 UPLOAD IMAGE FOR ANALYSIS
    </div>
    <div style='color:#555; font-size:0.85rem; margin-top:0.5rem;'>
        Supported: JPG · PNG · WEBP
    </div>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("",
               type=["jpg", "jpeg", "png", "webp"],
               label_visibility="collapsed")

if uploaded:
    image = Image.open(uploaded).convert("RGB")

    st.markdown("<div class='glowing-divider'></div>",
                unsafe_allow_html=True)

    # Show image centered
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        st.image(image, caption="📸 Uploaded Image",
                 use_column_width=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # Analyzing animation
    with st.spinner("🔬 Scanning image with all 3 AI models..."):
        time.sleep(0.5)
        results = predict(image, models_dict)

    st.markdown("<div class='glowing-divider'></div>",
                unsafe_allow_html=True)
    st.markdown("""
    <div style='font-family:Orbitron,monospace; font-size:1.2rem;
                color:#00f5ff; letter-spacing:3px; margin:1rem 0;
                text-align:center;'>
        📊 MODEL PREDICTIONS
    </div>
    """, unsafe_allow_html=True)

    # Show each model result
    cols = st.columns(3)
    model_icons = {
        "VGG16":           "🧠",
        "EfficientNet-B3": "⚡",
        "LeViT-128s":      "🔮"
    }

    for i, (model_name, res) in enumerate(results.items()):
        label = res["label"]
        conf  = res["confidence"]
        card_class = "fake-card" if label == "FAKE" else "real-card"
        pred_class  = "pred-fake" if label == "FAKE" else "pred-real"
        emoji = "🔴" if label == "FAKE" else "🟢"
        icon  = model_icons.get(model_name, "🤖")

        with cols[i]:
            st.markdown(f"""
            <div class='model-card {card_class}'>
                <div class='model-name'>
                    {icon} {model_name}
                </div>
                <hr style='border-color:rgba(255,255,255,0.1);
                            margin:0.8rem 0;'>
                <div style='text-align:center;'>
                    <div class='{pred_class}'>
                        {emoji} {label}
                    </div>
                    <div class='confidence' style='margin-top:0.5rem;'>
                        {conf*100:.2f}% confident
                    </div>
                </div>
                <hr style='border-color:rgba(255,255,255,0.1);
                            margin:0.8rem 0;'>
                <div style='color:#888; font-size:0.8rem;
                            margin-bottom:0.3rem;'>
                    FAKE probability
                </div>
            </div>
            """, unsafe_allow_html=True)
            st.progress(int(res["fake_prob"] * 100))
            st.markdown(f"""
            <div style='color:#888; font-size:0.8rem;
                        margin-bottom:0.3rem; margin-top:0.5rem;'>
                REAL probability
            </div>
            """, unsafe_allow_html=True)
            st.progress(int(res["real_prob"] * 100))

    # Final Verdict
    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("<div class='glowing-divider'></div>",
                unsafe_allow_html=True)
    st.markdown("""
    <div style='font-family:Orbitron,monospace; font-size:1.2rem;
                color:#ffff00; letter-spacing:3px; margin:1rem 0;
                text-align:center;'>
        🏆 FINAL VERDICT
    </div>
    """, unsafe_allow_html=True)

    votes = [r["label"] for r in results.values()]
    final = max(set(votes), key=votes.count)
    fake_votes = votes.count("FAKE")
    real_votes = votes.count("REAL")

    if final == "FAKE":
        st.markdown(f"""
        <div class='verdict-fake'>
            🔴 AI GENERATED IMAGE<br>
            <span style='font-size:1rem; color:#ff9999;'>
                {fake_votes}/3 models detected FAKE
            </span>
        </div>
        """, unsafe_allow_html=True)
    else:
        st.markdown(f"""
        <div class='verdict-real'>
            🟢 REAL AUTHENTIC IMAGE<br>
            <span style='font-size:1rem; color:#99ffcc;'>
                {real_votes}/3 models detected REAL
            </span>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # Detailed breakdown
    with st.expander("📊 Detailed Probability Breakdown"):
        for model_name, res in results.items():
            st.markdown(f"**{model_icons.get(model_name)} {model_name}**")
            col1, col2 = st.columns(2)
            with col1:
                st.markdown(f"🔴 FAKE: `{res['fake_prob']*100:.2f}%`")
                st.progress(int(res['fake_prob'] * 100))
            with col2:
                st.markdown(f"🟢 REAL: `{res['real_prob']*100:.2f}%`")
                st.progress(int(res['real_prob'] * 100))
            st.markdown("---")

else:
    st.markdown("<br>", unsafe_allow_html=True)
    # Welcome screen
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("""
        <div class='model-card' style='text-align:center;'>
            <div style='font-size:2.5rem;'>🧠</div>
            <div class='model-name' style='margin-top:0.5rem;'>VGG16</div>
            <div style='color:#888; font-size:0.85rem; margin-top:0.5rem;'>
                Deep CNN with 16 layers.<br>Highest accuracy model.
            </div>
        </div>""", unsafe_allow_html=True)
    with col2:
        st.markdown("""
        <div class='model-card' style='text-align:center;'>
            <div style='font-size:2.5rem;'>⚡</div>
            <div class='model-name' style='margin-top:0.5rem;'>
                EfficientNet-B3
            </div>
            <div style='color:#888; font-size:0.85rem; margin-top:0.5rem;'>
                Compound scaled CNN.<br>Fast and efficient.
            </div>
        </div>""", unsafe_allow_html=True)
    with col3:
        st.markdown("""
        <div class='model-card' style='text-align:center;'>
            <div style='font-size:2.5rem;'>🔮</div>
            <div class='model-name' style='margin-top:0.5rem;'>
                LeViT-128s
            </div>
            <div style='color:#888; font-size:0.85rem; margin-top:0.5rem;'>
                Hybrid Transformer.<br>Attention-based detection.
            </div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    st.markdown("""
    <div style='text-align:center; color:#555;
                font-family:Rajdhani,sans-serif;
                font-size:1rem; letter-spacing:2px;'>
        ⬆️ UPLOAD AN IMAGE ABOVE TO BEGIN ANALYSIS
    </div>
    """, unsafe_allow_html=True)
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ Beautiful app.py written!")

import subprocess, threading, time, re

def run_streamlit():
    subprocess.run(
        ["streamlit", "run", "app.py",
         "--server.port", "8501",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
print("⏳ Starting Streamlit... waiting 8 seconds...")
time.sleep(8)

process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
print("⏳ Starting tunnel... waiting 10 seconds...")
time.sleep(10)

output = ""
for _ in range(30):
    line = process.stderr.readline().decode("utf-8")
    output += line
    match = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        print("=" * 55)
        print("  ✅ Your Beautiful App is LIVE!")
        print(f"  🌐 Open this URL:  {url}")
        print("  ⚠️  Keep this cell running!")
        print("=" * 55)
        break
else:
    print("Could not find URL. Output:")
    print(output)

process.wait()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 130.2 MB/s eta 0:00:00
✅ Beautiful app.py written!
⏳ Starting Streamlit... waiting 8 seconds...
⏳ Starting tunnel... waiting 10 seconds...
  ✅ Your Beautiful App is LIVE!
  🌐 Open this URL:  https://excuse-networks-anywhere-filling.trycloudflare.com
  ⚠️  Keep this cell running!
